<a href="https://colab.research.google.com/github/SridharS-Square/Agentic_AI_Workshop/blob/main/Building%20Advanced%20Al%20Agents%20with%20AutoGen/Bill_Managing_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pyautogen google-generativeai pillow

In [ ]:
# main_script.py

import os
import io
import base64
import json
import google.generativeai as genai
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from PIL import Image

# --- Configuration ---

def setup_api():
    """Configures the Google Generative AI API."""
    # It's recommended to use environment variables or a secure key management service.
    # For this example, we'll use a placeholder.
    api_key = os.environ.get("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")
    if api_key == "YOUR_GOOGLE_API_KEY_HERE":
        print("⚠️ Warning: Using a placeholder API key. Please set the GOOGLE_API_KEY environment variable.")

    genai.configure(api_key=api_key)
    return api_key

API_KEY = setup_api()

MODEL_CONFIG = {
    "config_list": [
        {
            "model": "gemini-1.5-flash",
            "api_key": API_KEY,
            "api_type": "google",
        }
    ],
    "temperature": 0.7,
}


# --- Helper Functions ---

def get_mock_data(data_type="generic"):
    """Provides sample expense data for demonstration purposes."""
    sample_data = {
        "groceries": {
            "category": "Groceries",
            "items": ["Organic Milk - $4.50", "Whole Wheat Bread - $3.25", "Free-Range Eggs - $5.00", "Avocado - $2.25"],
            "total": 15.00,
        },
        "dining": {
            "category": "Dining Out",
            "items": ["Margherita Pizza - $18.00", "Craft Beer - $8.50", "Service Charge - $3.00"],
            "total": 29.50,
        },
        "shopping": {
            "category": "Retail Shopping",
            "items": ["Graphic T-Shirt - $30.00", "Denim Jeans - $75.00", "Sneakers - $90.00"],
            "total": 195.00,
        }
    }
    return sample_data.get(data_type.lower(), sample_data["groceries"])

def encode_image_for_model(file_path):
    """Encodes an image file into a base64 string for model consumption."""
    try:
        with Image.open(file_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Resize image to prevent it from being too large
            img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)

            byte_buffer = io.BytesIO()
            img.save(byte_buffer, format="JPEG")
            return base64.b64encode(byte_buffer.getvalue()).decode('utf-8')
    except FileNotFoundError:
        print(f"❌ Error: The file '{file_path}' was not found.")
        return None
    except Exception as e:
        print(f"❌ An error occurred while processing the image: {e}")
        return None

# --- Agent Definitions ---

# 1. Client Agent (Represents the User)
client_agent = UserProxyAgent(
    name="Client_Agent",
    human_input_mode="NEVER",
    code_execution_config={"use_docker": False},
    system_message="""You are a client representative. Your job is to provide the initial task, including any receipt images or data, to the team. You will start the process and await the final report."""
)

# 2. Receipt Parser Agent
receipt_parser = AssistantAgent(
    name="Receipt_Parser",
    llm_config=MODEL_CONFIG,
    system_message="""As the Receipt Parser, your specialty is analyzing receipts, whether from an image or text.
    Your tasks are:
    1.  Diligently extract every line item and its price.
    2.  Determine the most fitting category for the expense (e.g., Groceries, Dining Out, Utilities, Retail Shopping).
    3.  Present a clean, structured output with the category, a list of items, and the category's total cost.

    Output Format Example:
    - Category: [Determined Category]
    - Items: [List of all items and their prices]
    - Category Total: [Total amount for this category]
    """
)

# 3. Financial Analyst Agent
financial_analyst = AssistantAgent(
    name="Financial_Analyst",
    llm_config=MODEL_CONFIG,
    system_message="""You are the Financial Analyst. You receive structured data from the Receipt Parser.
    Your responsibilities are:
    1.  Consolidate all provided data to calculate the total expenditure.
    2.  Create a percentage breakdown of spending by category.
    3.  Pinpoint the category with the highest spending.
    4.  Deliver sharp, actionable insights to help with budgeting and financial awareness.

    Final Report Format:
    - Total Spend: [Total Amount]
    - Spending Breakdown: [List Category: Amount (Percentage)]
    - Top Spending Area: [Category with the highest spend]
    - Key Insights: [Actionable advice and observations]
    """
)


# --- Group Chat and Manager Setup ---

expense_chat_group = GroupChat(
    agents=[client_agent, receipt_parser, financial_analyst],
    messages=[],
    max_round=12,
    speaker_selection_method="round_robin"
)

chat_coordinator = GroupChatManager(
    groupchat=expense_chat_group,
    llm_config=MODEL_CONFIG,
    system_message="""You are the Chat Coordinator. Your job is to manage the workflow between the agents.
    Ensure the process flows logically: from the Client Agent's request, to the Receipt Parser's analysis, and finally to the Financial Analyst's summary. Keep the conversation on track and efficient."""
)


# --- Main Application Logic ---

def run_agentic_workflow():
    """Initiates and runs the multi-agent expense management system."""
    print("\n" + "="*50)
    print("🤖 Welcome to the Agentic Expense Manager 🤖")
    print("="*50)

    print("\nHow would you like to provide the bill?")
    print("  1. Provide a path to a bill image.")
    print("  2. Use sample data (e.g., 'groceries', 'dining', 'shopping').")

    task_content = None
    choice = input("\nEnter your choice (1 or 2): ").strip()

    if choice == '1':
        image_path = input("Enter the full path to your image file: ").strip()
        base64_image = encode_image_for_model(image_path)
        if base64_image:
            print(f"✅ Image '{image_path}' successfully encoded.")
            task_content = [
                {"type": "text", "text": "Please process the attached receipt image, extract the items, categorize them, and provide a financial summary."},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]
        else:
            print("Could not process image. Defaulting to sample 'groceries' data.")
            task_content = f"Here is the bill data to process: {json.dumps(get_mock_data('groceries'), indent=2)}"

    else:
        bill_kind = input("Enter sample data type (groceries/dining/shopping): ").strip()
        mock_data = get_mock_data(bill_kind)
        print(f"✅ Using sample '{bill_kind}' data.")
        task_content = f"Here is the bill data to process: {json.dumps(mock_data, indent=2)}"

    print("\n" + "~"*50)
    print("🚀 Initiating analysis... Please wait.")
    print("~"*50 + "\n")

    client_agent.initiate_chat(
        recipient=chat_coordinator,
        message=task_content
    )

def generate_basic_report():
    """A simple, non-AI fallback function for expense reporting."""
    print("\n" + "="*50)
    print("📊 Basic Expense Report 📊")
    print("="*50)

    spending_buckets = {
        "Utilities": 95.50,
        "Groceries": 85.20,
        "Transport": 60.00,
        "Entertainment": 45.00,
        "Shopping": 210.00
    }

    grand_total = sum(spending_buckets.values())
    top_category = max(spending_buckets, key=spending_buckets.get)

    print(f"\n💰 Total Expenditure: ${grand_total:.2f}")
    print("\n📈 Category Breakdown:")

    for category, amount in spending_buckets.items():
        percent = (amount / grand_total) * 100 if grand_total > 0 else 0
        print(f"  - {category:<15} ${amount:>7.2f} ({percent:.1f}%)")

    print(f"\n🎯 Highest Spending Category: {top_category} (${spending_buckets[top_category]:.2f})")
    print("\n💡 Key Insights:")
    if spending_buckets[top_category] > grand_total * 0.4:
        print(f"  - Your spending on {top_category} is a significant portion of your total expenses. Reviewing this area could offer savings opportunities.")
    print("  - Regularly tracking expenses helps in identifying trends and managing your budget effectively.")


if __name__ == "__main__":
    try:
        run_agentic_workflow()
    except Exception as e:
        print(f"\n🚨 An unexpected error occurred in the agentic system: {e}")
        print("...Switching to the basic reporting tool...")
        generate_basic_report()
    finally:
        print("\n" + "="*50)
        print("✅ Process Finished.")
        print("="*50)

```